# 04. GRPO DAPO and Verifiable Reward Reasoning

- **Category:** 08_learning_paradigms_rl_and_alignment / 03_llm_alignment_and_reasoning
- **Lineage & Citations:** Schulman et al. (2017 Proximal Policy Optimization); Shao et al. (2024 DeepSeekMath / Group Relative Policy Optimization); DeepSeek-AI (2025 DeepSeek-R1 & DeepSeek-R1-Zero).
- **Purpose:** First-principles mathematical implementation and empirical verification of verifiable reasoning alignment: Group Relative Policy Optimization (GRPO) advantage estimation, Schulman's k3 unbiased KL estimator, token-level PPO clipped surrogate optimization, and deterministic rule-based rewards.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Direct Preference DPO IPO KTO SimPO DNO and SPO](03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.ipynb) $\to$ `🔗 Downstream Syntheses:` [05. Test Time Compute PRM Best of N and MCTS](05_test_time_compute_prm_best_of_n_and_mcts.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never use GRPO on subjective tasks lacking objective verifiable ground-truth rules (e.g. creative writing, style imitation); without objective verification, group relative advantages fail to distinguish factual reasoning from hallucinated rhetoric. Never execute GRPO with group size $G=1$; normalizing across a group requires $G \ge 2$ (typically $G \in [4, 16]$) to estimate empirical mean and variance.

## Mathematical Foundations & Governing Equations

### 1. GRPO Group Relative Advantage Estimation
$$
\mu = \frac{1}{G} \sum_{i=1}^G R_i, \quad \sigma = \sqrt{\frac{1}{G} \sum_{i=1}^G (R_i - \mu)^2 + \epsilon}
$$
$$
\hat{A}_i = \frac{R_i - \mu}{\sigma}
$$

### 2. Schulman's $k_3$ Unbiased Low-Variance KL Estimator
$$
\Delta_{i, t} = \ln \pi_\theta(o_{i, t}) - \ln \pi_{\text{ref}}(o_{i, t})
$$
$$
D_{\text{KL}}^{k_3}(\pi_\theta \| \pi_{\text{ref}}) = e^{-\Delta_{i, t}} - 1 + \Delta_{i, t} \ge 0
$$

### 3. Clipped GRPO Surrogate Objective
$$
\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^G \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \left[ \min\left( r_{i, t}(\theta) \hat{A}_i, \, \text{clip}(r_{i, t}(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_i \right) - \beta D_{\text{KL}}^{k_3}(\pi_\theta \| \pi_{\text{ref}})_{i, t} \right]
$$

### 4. Composite Verifiable Reward
$$
R_i = w_{\text{acc}} \cdot \mathbb{I}(\text{Accuracy Valid}) + w_{\text{format}} \cdot \mathbb{I}(\text{Format Tag Compliant})
$$

### Asymptotic Time & Memory Complexity
- Group Advantage Normalization: $\mathcal{O}(B \cdot G)$ time and memory.
- $k_3$ KL Divergence: $\mathcal{O}(B \cdot G \cdot T)$ time and memory.
- Clipped GRPO Loss: $\mathcal{O}(B \cdot G \cdot T)$ time and memory (Zero critic memory!).
- Composite Reward Verification: $\mathcal{O}(B \cdot G)$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def composite_verifiable_reward(
    accuracy_flags: torch.Tensor,
    format_flags: torch.Tensor,
    acc_weight: float = 1.0,
    format_weight: float = 0.5,
) -> torch.Tensor:
    return acc_weight * accuracy_flags + format_weight * format_flags

def grpo_group_advantages(
    rewards: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    mean = torch.mean(rewards, dim=-1, keepdim=True)
    std = torch.std(rewards, dim=-1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)

def k3_kl_divergence(
    log_probs_policy: torch.Tensor,
    log_probs_ref: torch.Tensor,
) -> torch.Tensor:
    delta = log_probs_policy - log_probs_ref
    return torch.exp(-delta) - 1.0 + delta

def grpo_clipped_loss(
    log_probs_policy: torch.Tensor,
    log_probs_old: torch.Tensor,
    advantages: torch.Tensor,
    kl_div: torch.Tensor,
    clip_eps: float = 0.2,
    beta: float = 0.04,
) -> torch.Tensor:
    ratio = torch.exp(log_probs_policy - log_probs_old)
    adv = advantages.unsqueeze(-1)
    surr1 = ratio * adv
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv
    ppo_obj = torch.min(surr1, surr2)
    token_obj = ppo_obj - beta * kl_div
    return -torch.mean(token_obj)

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

rew = composite_verifiable_reward(weights["accuracy_flags"], weights["format_flags"], acc_weight=1.0, format_weight=0.5)
err_rew = float(torch.max(torch.abs(rew - weights["rewards_ref"])))
assert err_rew < 1e-5, f"Composite rewards parity error: {err_rew:.2e}"

adv = grpo_group_advantages(weights["rewards_ref"], eps=1e-6)
err_adv = float(torch.max(torch.abs(adv - weights["advantages_ref"])))
assert err_adv < 1e-5, f"GRPO advantages parity error: {err_adv:.2e}"

kl = k3_kl_divergence(weights["log_probs_policy"], weights["log_probs_ref"])
err_kl = float(torch.max(torch.abs(kl - weights["kl_div_ref"])))
assert err_kl < 1e-5, f"K3 KL divergence parity error: {err_kl:.2e}"

loss = grpo_clipped_loss(weights["log_probs_policy"], weights["log_probs_old"], weights["advantages_ref"], weights["kl_div_ref"], clip_eps=0.2, beta=0.04)
err_loss = float(torch.abs(loss - weights["grpo_loss_ref"]))
assert err_loss < 1e-5, f"GRPO loss parity error: {err_loss:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_rew, err_adv, err_kl, err_loss):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_rew = benchmark_fn(
    lambda: composite_verifiable_reward(weights["accuracy_flags"], weights["format_flags"]),
    iters=100,
    name="Composite Verifiable Rewards",
)
b_adv = benchmark_fn(
    lambda: grpo_group_advantages(weights["rewards_ref"]),
    iters=100,
    name="GRPO Group Advantages",
)
b_kl = benchmark_fn(
    lambda: k3_kl_divergence(weights["log_probs_policy"], weights["log_probs_ref"]),
    iters=100,
    name="K3 KL Divergence",
)
b_loss = benchmark_fn(
    lambda: grpo_clipped_loss(weights["log_probs_policy"], weights["log_probs_old"], weights["advantages_ref"], weights["kl_div_ref"]),
    iters=100,
    name="GRPO Clipped Loss",
)

print(format_benchmark_table([b_rew, b_adv, b_kl, b_loss]))

In [ ]:
# Failure Mode Contrast: Raw Log-Ratio vs Schulman's Non-Negative k3 KL Estimator
import math
# Suppose active policy has drifted from reference: pi = 0.8, ref = 0.2
log_p = torch.tensor([math.log(0.8)])
log_r = torch.tensor([math.log(0.2)])

# Case 1: Naive difference estimator
naive_diff = (log_p - log_r).item()  # log(4) = 1.3863

# Case 2: Schulman's k3 estimator = exp(-delta) - 1 + delta
k3_est = k3_kl_divergence(log_p, log_r).item()
# exp(-1.3863) - 1 + 1.3863 = 0.25 - 1.0 + 1.3863 = 0.6363

print(f"Naive Log-Ratio Difference: {naive_diff:.4f} (Unbounded, high variance, can be negative under token inversion!)")
print(f"Schulman k3 KL Estimator:   {k3_est:.4f} (Strictly non-negative, minimum variance anchor!)")

assert k3_est > 0.0, f"k3 estimator must be strictly non-negative (got {k3_est})"
assert abs(k3_est - 0.6363) < 1e-3, f"Expected ~0.6363 (got {k3_est})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Critic-Free Group Normalization**: Normalizing trajectory rewards across $G$ parallel reasoning rollouts per prompt delivers variance reduction without a parameter-heavy value network.
2. **Ground-Truth Rule Verification**: Grounding rewards in execution or regex accuracy eliminates Goodhart's law and reward model hacking.
3. **Schulman's $k_3$ Robustness**: The non-negative $k_3$ estimator prevents policy collapse during long-horizon reasoning rollouts.
4. **Emergence of Test-Time Search**: DeepSeek-R1 demonstrated that training on pure verifiable rewards spontaneously discovers self-correction and multi-step verification traces.

---
### Cross-Domain Generalisation Matrix
- **Mathematical Reasoning (DeepSeek-R1)**: GRPO with boxed answer checking trains models to solve complex AIME competition math.
- **Compiler & Test-Guided Code Synthesis**: Python execution feedback guides policies to synthesize bug-free algorithms.
- **Automated Chemical Reaction Synthesizability**: Reaction feasibility verifiers guide generative molecular search.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_grpo_dapo_and_verifiable_reward_reasoning.md](../../../markdown/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.md)
- **PyTorch 2.x Module:** [04_grpo_dapo_and_verifiable_reward_reasoning.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.py)
- **Functional JAX Module:** [04_grpo_dapo_and_verifiable_reward_reasoning.py](../../../jax/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.py)
- **Native Rust Candle Track:** [04_grpo_dapo_and_verifiable_reward_reasoning.rs](../../../candle/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.rs)
- **Hardware Kernel (CPU / WGSL):** [04_grpo_dapo_and_verifiable_reward_reasoning.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.rs)
- **Golden SafeTensors:** [04_grpo_dapo_and_verifiable_reward_reasoning.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/04_grpo_dapo_and_verifiable_reward_reasoning.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Direct Preference DPO IPO KTO SimPO DNO and SPO](03_direct_preference_dpo_ipo_kto_simpo_dno_and_spo.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 05. Test Time Compute PRM Best of N and MCTS](05_test_time_compute_prm_best_of_n_and_mcts.ipynb) &rarr;